# Escala da busca de Dürr–Høyer (E0 numérico e E1, hipótese H4)

Compara o custo para encontrar a interdição ótima em três métodos que buscam **no mesmo espaço**, as interdições viáveis da variante de cardinalidade, F_k = {x : |x| ≤ k}, com N = |F_k|:

- **Dürr–Høyer** simulado no nível de consultas (simulador reduzido, validado contra o vetor de estado em `tests/test_grover.py`). Custo = consultas ao oráculo (iterações de Grover) + avaliações clássicas (candidatos medidos e conferidos), até medir o ótimo pela primeira vez;
- **ordem aleatória**: avaliações até encontrar o ótimo, com média teórica (N+1)/(M+1);
- **enumeração exaustiva**: N avaliações para certificar o ótimo.

**H4 (protocolo congelado em 06/11, 09h40):** o custo médio do Dürr–Høyer cresce como N^β com β ∈ [0,4; 0,6]; a ordem aleatória cresce com expoente próximo de 1. Critério: a estimativa pontual de β, reportada com IC 95% por *bootstrap* sobre instâncias.

O MILP é o gabarito do ótimo em todas as instâncias.

## 1. Imports, instâncias e constantes

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from qmfnip.evaluation import fit_scaling_exponent
from qmfnip.instances import Scenario, full_cut_cost, max_flow
from qmfnip.outputs import latest_experiment
from qmfnip.playground import ExperimentConfig, run_experiment
from qmfnip.plots import plot_scaling
from qmfnip.solvers import BruteForceSolver, DurrHoyerSolver, MILPSolver, RandomOrderSolver

In [ ]:
SCENARIO = Scenario(
    "layered",
    sizes=(2, 3, 4, 5),
    budgets=(1, 2, 3, 4),
    seeds=(0, 1, 2),
    params={"cost_range": (1, 1)},
    skip_degenerate=True,
)
SOLVERS = [
    DurrHoyerSolver(query_budget="unbounded"),
    RandomOrderSolver(),
    BruteForceSolver(label="enumeration"),
]
REPETITIONS = 200
TIME_LIMIT = 3600.0
BETA_INTERVAL = (0.4, 0.6)
N_BOOT = 2000
EXPERIMENT_TAG = "durr_hoyer-random_order-enumeration__layered__n6-27__runs200"
RERUN = False
LABELS = {
    "durr_hoyer": "Dürr–Høyer (queries + evaluations)",
    "random_order": "random order (evaluations)",
    "enumeration": "exhaustive enumeration (evaluations)",
}
CONFIG = ExperimentConfig(
    scenario=SCENARIO,
    solvers=SOLVERS,
    repetitions=REPETITIONS,
    time_limit=TIME_LIMIT,
    reference=MILPSolver(),
    reference_time_limit=600.0,
)
instances = list(SCENARIO.instances())
len(instances)

## 2. Execução dos algoritmos e testes

Conferência de degenerescência antes de rodar: com custo unitário, o corte que zera o fluxo custa κ (a largura da rede), e `skip_degenerate` já garante k < κ, logo ótimo > 0. Falta conferir o outro extremo (ótimo < fluxo original), com o MILP.

In [ ]:
checks = []
for instance in instances:
    milp = MILPSolver().solve(instance, seed=0, time_limit=60)
    checks.append(
        {
            "instance_id": instance.instance_id,
            "k": instance.budget,
            "kappa": full_cut_cost(instance),
            "flow_without_interdiction": max_flow(instance),
            "optimal_flow": milp.objective,
            "certified": milp.certified_optimal,
        }
    )
checks = pd.DataFrame(checks)
checks["non_degenerate"] = (checks["optimal_flow"] > 0) & (checks["optimal_flow"] < checks["flow_without_interdiction"])
assert checks["certified"].all() and checks["non_degenerate"].all() and (checks["k"] < checks["kappa"]).all()
checks

Execução. Se já houver um experimento concluído com a mesma configuração, ele é reaproveitado (`RERUN = False`); se houver um interrompido, ele é retomado. A execução completa leva ~30 min nesta máquina, quase toda no cálculo dos fluxos das ~974 mil interdições de k = 4.

In [ ]:
try:
    previous = latest_experiment(contains=EXPERIMENT_TAG)
    previous_status = previous.read_json("manifest.json").get("status")
except FileNotFoundError:
    previous, previous_status = None, None
if previous_status == "completed" and not RERUN:
    output = previous
else:
    output = run_experiment(CONFIG, resume=None if previous is None or RERUN else previous.path, verbose=False)
output.path.name

## 3. Avaliação dos resultados, gráficos e tabelas

In [ ]:
runs = output.load_runs()
references = pd.DataFrame(output.read_records("references.jsonl"))
stored = output.load_instances()
flags = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}
for column in ("found_optimum", "within_query_budget", "non_degenerate"):
    runs[column] = pd.to_numeric(runs[column].map(lambda value: flags.get(value, value)), errors="coerce")
runs["cost_to_optimum"] = np.where(
    runs["solver"] == "enumeration",
    runs["classical_evaluations"],
    runs["queries_to_optimum"] + runs["evaluations_to_optimum"],
)
runs["within_cost_budget"] = np.where(
    runs["solver"] == "durr_hoyer", (runs["cost_to_optimum"] <= runs["query_budget"]).astype(float), np.nan
)
print(f"runs: {len(runs)}  instances: {runs['instance_id'].nunique()}  solvers: {sorted(runs['solver'].unique())}")

### Aceite

1. A referência do MILP é certificada em todas as instâncias.
2. O ótimo do espaço enumerado (`enumeration`) coincide com o do MILP em todas.
3. Nenhuma instância é degenerada.
4. Toda execução que declara ter achado o ótimo tem gap 0 (fluxo recalculado de forma independente).

In [ ]:
enumeration = runs[runs["solver"] == "enumeration"].drop_duplicates("instance_id")
acceptance = {
    "milp_certified": f"{int(references['certified'].sum())}/{len(references)}",
    "enumeration_matches_milp": f"{int((enumeration['flow_after_interdiction'] == enumeration['optimal_flow']).sum())}/{len(enumeration)}",
    "non_degenerate": f"{int(runs.drop_duplicates('instance_id')['non_degenerate'].sum())}/{runs['instance_id'].nunique()}",
    "found_runs_with_zero_gap": f"{int((runs.loc[runs['found_optimum'] == 1, 'gap'] == 0).sum())}/{int((runs['found_optimum'] == 1).sum())}",
}
output.write_json("acceptance.json", acceptance)
acceptance

### Tabela T1: instâncias

In [ ]:
t1 = (
    runs[runs["solver"] == "enumeration"]
    .drop_duplicates("instance_id")[
        ["instance_id", "instance_seed", "n_nodes", "n_arcs", "budget", "search_space_size", "optimal_count",
         "flow_without_interdiction", "optimal_flow", "enumeration_time_s"]
    ]
    .merge(references[["instance_id", "wall_time_s"]].rename(columns={"wall_time_s": "milp_time_s"}), on="instance_id")
)
t1.insert(5, "kappa", [full_cut_cost(stored[name]) for name in t1["instance_id"]])
t1 = t1.rename(columns={"budget": "k", "search_space_size": "N", "optimal_count": "M"}).sort_values(["N", "instance_seed"])
output.write_table("t1_instances", t1)
t1

### Custo médio por instância e método

In [ ]:
per_instance = (
    runs.groupby(["solver", "instance_id"], sort=False)
    .agg(
        N=("search_space_size", "first"),
        M=("optimal_count", "first"),
        k=("budget", "first"),
        n_arcs=("n_arcs", "first"),
        repetitions=("cost_to_optimum", "size"),
        mean_cost=("cost_to_optimum", "mean"),
        std_cost=("cost_to_optimum", "std"),
        mean_queries=("queries_to_optimum", "mean"),
        mean_evaluations=("evaluations_to_optimum", "mean"),
        found_rate=("found_optimum", "mean"),
        within_budget_rate=("within_query_budget", "mean"),
        within_cost_budget_rate=("within_cost_budget", "mean"),
        query_budget=("query_budget", "first"),
    )
    .reset_index()
)
per_instance["N_over_M"] = per_instance["N"] / per_instance["M"]
per_instance["sqrt_N"] = np.sqrt(per_instance["N"])
per_instance["cost_over_sqrt_N"] = per_instance["mean_cost"] / per_instance["sqrt_N"]
per_instance["random_order_theory"] = (per_instance["N"] + 1) / (per_instance["M"] + 1)
output.write_table("per_instance", per_instance)
per_instance[per_instance["solver"] == "durr_hoyer"].sort_values("N")[
    ["instance_id", "N", "M", "mean_cost", "std_cost", "mean_queries", "mean_evaluations", "cost_over_sqrt_N",
     "within_budget_rate", "within_cost_budget_rate", "query_budget"]
]

Taxa de acerto do Dürr–Høyer dentro do orçamento teórico de consultas, 22,5√N + 1,4 log₂²N (o teorema garante ≥ 1/2). Duas leituras: só consultas ≤ Q (`within_budget_rate`) e custo total, consultas + avaliações ≤ Q (`within_cost_budget_rate`, a leitura conservadora usada no artigo: na contabilidade de Dürr e Høyer cada preparação custa log₂N passos, então o nosso custo nunca passa do tempo deles). Como cada execução para ao medir o ótimo, a execução acerta dentro do orçamento quando as consultas até o ótimo não passam dele; o resultado é o mesmo de uma execução que parasse ao esgotar o orçamento, porque a sequência sorteada é idêntica até aquele ponto.

In [ ]:
dh = per_instance[per_instance["solver"] == "durr_hoyer"]
budget_summary = {
    "min_within_cost_budget_rate": float(dh["within_cost_budget_rate"].min()),
    "mean_within_cost_budget_rate": float(dh["within_cost_budget_rate"].mean()),
    "max_mean_cost_over_budget": float((dh["mean_cost"] / dh["query_budget"]).max()),
    "min_within_budget_rate": float(dh["within_budget_rate"].min()),
    "mean_within_budget_rate": float(dh["within_budget_rate"].mean()),
    "instances_below_one_half": int((dh["within_budget_rate"] < 0.5).sum()),
    "max_mean_queries_over_budget": float((dh["mean_queries"] / dh["query_budget"]).max()),
}
output.write_json("query_budget_summary.json", budget_summary)
budget_summary

Ordem aleatória contra a média teórica (N+1)/(M+1): razão por instância (deve ficar perto de 1).

In [ ]:
ro = per_instance[per_instance["solver"] == "random_order"]
(ro["mean_cost"] / ro["random_order_theory"]).describe()

### Tabela T2: expoentes ajustados (regressão log-log, IC 95% por *bootstrap* sobre instâncias)

In [ ]:
fits = pd.concat(
    [
        fit_scaling_exponent(per_instance, "N", "mean_cost", n_boot=N_BOOT, seed=0),
        fit_scaling_exponent(per_instance, "N_over_M", "mean_cost", n_boot=N_BOOT, seed=0),
    ],
    ignore_index=True,
)
dh_fit = fits[(fits["solver"] == "durr_hoyer") & (fits["x"] == "N")].iloc[0]
h4 = {
    "beta": float(dh_fit["beta"]),
    "ci_low": float(dh_fit["ci_low"]),
    "ci_high": float(dh_fit["ci_high"]),
    "interval": list(BETA_INTERVAL),
    "point_estimate_inside": bool(BETA_INTERVAL[0] <= dh_fit["beta"] <= BETA_INTERVAL[1]),
    "ci_inside": bool(BETA_INTERVAL[0] <= dh_fit["ci_low"] and dh_fit["ci_high"] <= BETA_INTERVAL[1]),
}
h4["verdict"] = "not refuted" if h4["point_estimate_inside"] else "refuted"
output.write_table("t2_scaling_fits", fits)
output.write_json("h4.json", h4)
display(fits.round(4))
h4

### Figura F3: custo até o ótimo × N (log-log)

In [ ]:
figure = plot_scaling(
    per_instance,
    "N",
    "mean_cost",
    fits=fits[fits["x"] == "N"],
    labels=LABELS,
    xlabel="feasible interdictions N = |F_k|",
    ylabel="mean cost to the optimum",
    path=output.figure_path("f3_scaling"),
)

In [ ]:
figure = plot_scaling(
    per_instance,
    "N_over_M",
    "mean_cost",
    fits=fits[fits["x"] == "N_over_M"],
    labels=LABELS,
    xlabel="N / M",
    ylabel="mean cost to the optimum",
    path=output.figure_path("f3_scaling_n_over_m"),
)

### Contexto clássico (E4): tempo de relógio do MILP e da enumeração viável nas mesmas instâncias

In [ ]:
t5 = t1.groupby(["n_nodes", "n_arcs", "k", "N"], as_index=False).agg(
    milp_median_s=("milp_time_s", "median"),
    milp_max_s=("milp_time_s", "max"),
    enumeration_median_s=("enumeration_time_s", "median"),
    enumeration_max_s=("enumeration_time_s", "max"),
)
output.write_table("t5_classical_context", t5)
t5